---
# **Suplementos Alimentares**
---


🎯 **Objetivo:** Verificar qual das três fórmulas de proteína em pó (Fórumula 1, 2 ou 3) proporciona maior ganho de massa muscular em atletas?


---
**Variáveis:**

- `id_produto`: Código identificador do suplemento (Fórmula 1, 2 ou 3). 
- `id_atleta`: Código identificador do atleta que participou do estudo. 
- `ganho_massa`: Quantidade de massa muscular ganha (em kg) após 8 semanas de uso. 
- `idade`: Idade do atleta. 
- `frequencia_treino`: Número médio de treinos semanais do atleta. 
---

Desafio Estatística com Python - Teste de hipóteses
Squad Nina da Hora | Bootcamp Data Analytics 2026.1

In [ ]:
# ==============================
# IMPORTACOES
# ==============================

import math
import numpy as np
import pandas as pd

from IPython.display import display, Markdown

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Estatística
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Configuração visual
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

paleta = "flare"
cores = sns.color_palette(paleta, n_colors=2)

In [ ]:
# ==============================
# IMPORTACAO DO DATASET 
# ==============================

arquivo = 'bd_suplementos'
url = f'https://raw.githubusercontent.com/Squad-Nina-da-Hora/wmc-desafio-suplementos/main/{arquivo}.csv'
df = pd.read_csv(url)

In [ ]:
# ==============================
# PERFIL DO DATASET
# ==============================

linhas = df.shape[0]
colunas = df.shape[1]
print(f"O dataset possui {linhas} linhas e {colunas} colunas.")

display(Markdown("---"))

info_df = pd.DataFrame({
    'Coluna': df.columns,
    'Tipo': df.dtypes.values,
    'Não nulos': df.count().values,
    'Nulos': df.isnull().sum().values
})

info_df

### Questão 1 — Análise Exploratória dos Dados (EDA)

In [ ]:
# Tabela descritiva do ganho de massa para cada suplemento

sup_massa = df.groupby('id_produto')['ganho_massa'].agg(['mean', 'median', 'std', 'count'])
sup_massa.columns = ['Média de Ganho (kg)', 'Mediana de Ganho (kg)', 'Desvio Padrão (kg)', 'Tamanho da Amostra']

sup_massa = sup_massa.round(4)
sup_massa

In [ ]:
# Ganho de massa para cada suplemento
# Visual da distribuição reorganizado e com eixos padronizados

# Encontra os limites do ganho de massa para a padronização das escalas

ganho_min = df['ganho_massa'].min() - 0.5
ganho_max = df['ganho_massa'].max() + 0.5

limites_atleta = 20

fig, axes = plt.subplots(2, 3, figsize=(20, 14))

produto = sorted(df['id_produto'].unique())

for i, supl in enumerate(produto):
  df_supl = df[df['id_produto'] == supl]
  cor_atual = sns.color_palette('flare', 3)[i]

  # --- Histogramas ---
  sns.histplot(data=df_supl, x='ganho_massa', kde=True, color=cor_atual, stat='count', kde_kws={'cut': 3},ax=axes[0, i])
  axes[0, i].set_title(f'Histograma: {supl}', fontsize=12, weight='bold')
  axes[0, i].set_xlabel('Ganho de Massa (kg)')
  axes[0, i].set_ylabel('Número de atletas')
  
  # Padronizando os eixos X e Y para todos os histogramas
  axes[0, i].set_xlim(ganho_min, ganho_max)
  axes[0, i].set_ylim(0, limites_atleta) 

  # Adiciona os rótulos de contagem no histograma
  if axes[0, i].containers:
      axes[0, i].bar_label(axes[0, i].containers[0], padding=3, fontsize=9, weight='bold')

  # --- Boxplots ---
  sns.boxplot(data=df_supl, y='ganho_massa', color=cor_atual, ax=axes[1, i])
  sns.stripplot(data=df_supl, y='ganho_massa', color='black', alpha=0.4, size=5, jitter=0.1, ax=axes[1, i])
  axes[1, i].set_title(f'Boxplot: {supl}', fontsize=12, weight='bold')
  axes[1, i].set_ylabel('Ganho de Massa (kg)')
  axes[1, i].set_xlabel(supl)
  
  # Padronizando o eixo Y do Boxplot 
  axes[1, i].set_ylim(ganho_min, ganho_max)

# Ajusta o espaçamento 
plt.tight_layout(pad=3.0)
plt.subplots_adjust(hspace=0.4, wspace=0.3)
plt.show()


In [ ]:
# Outliers pelo método do IQR (geral)
q1 = df['ganho_massa'].quantile(0.25)
q3 = df['ganho_massa'].quantile(0.75)
iqr = q3 - q1

outliers = df[(df['ganho_massa'] < (q1 - 1.5 * iqr)) | (df['ganho_massa'] > (q3 + 1.5 * iqr))]
print(f"Total de outliers encontrados: {len(outliers)}")
outliers

# Fazendo análises extras com os dados do nossso dataframe
Seguindo essa ordem:

- Idade para cada Suplemento
- Frequência de treino para cada Suplemento
- Ganho de massa para frequência de treino
- Tabela descritiva do ganho de massa por idade
- Tabela descritiva do ganho de massa para o intervalo de idade criado
- Tabela descritiva da frequência de treino por idade
- Tabela descritiva da frequência de treino para o intervalo de idade criado


In [ ]:
# Tabela descritiva da idade para cada suplemento

sup_idade = df.groupby('id_produto')['idade'].agg(['mean', 'median', 'std', 'count'])
sup_idade.columns = ['Média de Idade', 'Mediana de Idade', 'Desvio Padrão', 'Tamanho da Amostra']

sup_idade = sup_idade.round(4)
sup_idade

In [ ]:
# Idade para cada suplemento
# Visual da distribuição da idade reorganizado e com eixos padronizados

# Encontra os limites da idade para a padronização das escalas

idade_min = df['idade'].min() - 2
idade_max = df['idade'].max() + 2

limite_atletas = 24

fig, axes = plt.subplots(2, 3, figsize=(20, 14))

produto = sorted(df['id_produto'].unique())

for i, supl in enumerate(produto):
  df_supl = df[df['id_produto'] == supl]
  cor_atual = sns.color_palette('flare', 3)[i]

  # --- Histogramas da Idade ---
  sns.histplot(data=df_supl, x='idade', kde=True, color=cor_atual, stat='count', kde_kws={'cut': 3}, ax=axes[0, i])
  axes[0, i].set_title(f'Histograma: {supl}', fontsize=12, weight='bold')
  axes[0, i].set_xlabel('Idade (anos)')
  axes[0, i].set_ylabel('Número de atletas')
  
  # Padronizando os eixos X e Y para todos os histogramas de idade
  axes[0, i].set_xlim(idade_min, idade_max)
  axes[0, i].set_ylim(0, limite_atletas)

  # Adiciona os rótulos de contagem no histograma
  if axes[0, i].containers:
      axes[0, i].bar_label(axes[0, i].containers[0], padding=3, fontsize=9, weight='bold')

  # --- Boxplots da Idade ---
  sns.boxplot(data=df_supl, y='idade', color=cor_atual, ax=axes[1, i])
  sns.stripplot(data=df_supl, y='idade', color='black', alpha=0.4, size=5, jitter=0.1, ax=axes[1, i])
  axes[1, i].set_title(f'Boxplot: {supl}', fontsize=12, weight='bold')
  axes[1, i].set_ylabel('Idade (anos)')
  axes[1, i].set_xlabel(supl)
  
  # Padronizando o eixo Y do Boxplot 
  axes[1, i].set_ylim(idade_min, idade_max)

# Ajusta os espaçamentos 
plt.tight_layout(pad=3.0)
plt.subplots_adjust(hspace=0.4, wspace=0.3)
plt.show()


In [ ]:
# Outliers pelo método do IQR (geral)
q1 = df['idade'].quantile(0.25)
q3 = df['idade'].quantile(0.75)
iqr = q3 - q1

outliers = df[(df['idade'] < (q1 - 1.5 * iqr)) | (df['idade'] > (q3 + 1.5 * iqr))]
print(f"Total de outliers encontrados: {len(outliers)}")
outliers

In [ ]:
# Tabela descritiva da frequência de treino para cada suplemento

sup_treino = df.groupby('id_produto')['frequencia_treino'].agg(['mean', 'median', 'std', 'count'])
sup_treino.columns = ['Média de Treinos', 'Mediana de Treinos', 'Desvio Padrão', 'Tamanho da Amostra']

sup_treino = sup_treino.round(4)
sup_treino

In [ ]:
# Frequência de treino para cada suplemento
# Visual da distribuição da frequência de treino por produto reorganizado e com eixos padronizados

# Encontra os limites da frequência de treino para a padronização das escalas

treino_min = df['frequencia_treino'].min() - 0.5
treino_max = df['frequencia_treino'].max() + 0.5

limite_atleta_freq = 25

fig, axes = plt.subplots(2, 3, figsize=(20, 14))

produto = sorted(df['id_produto'].unique())

for i, supl in enumerate(produto):
  df_supl = df[df['id_produto'] == supl]
  cor_atual = sns.color_palette('flare', 3)[i]

  # --- Histogramas das Frequências ---
  sns.histplot(data=df_supl, x='frequencia_treino', kde=True, color=cor_atual, stat='count', discrete=True, shrink=0.7, kde_kws={'cut': 3}, ax=axes[0, i])
  axes[0, i].set_title(f'Histograma: {supl}', fontsize=12, weight='bold')
  axes[0, i].set_xlabel('Frequência (Dias/Semana)')
  axes[0, i].set_ylabel('Número de atletas')
  
  # Padronizando os eixos X e Y para todos os histogramas de treino
  axes[0, i].set_xlim(treino_min, treino_max)
  axes[0, i].set_ylim(0, limite_atleta_freq)

  # Adiciona os rótulos de contagem no histograma
  if axes[0, i].containers:
      axes[0, i].bar_label(axes[0, i].containers[0], padding=3, fontsize=9, weight='bold')

  # --- Boxplots das frequências ---
  sns.boxplot(data=df_supl, y='frequencia_treino', color=cor_atual, ax=axes[1, i])
  sns.stripplot(data=df_supl, y='frequencia_treino', color='black', alpha=0.4, size=5, jitter=0.1, ax=axes[1, i])
  axes[1, i].set_title(f'Boxplot: {supl}', fontsize=12, weight='bold')
  axes[1, i].set_ylabel('Frequência (Dias/Semana)')
  axes[1, i].set_xlabel(supl)
  
  # Padronizando o eixo Y do Boxplot
  axes[1, i].set_ylim(treino_min, treino_max)

# Ajusta os espaçamentos
plt.tight_layout(pad=3.0)
plt.subplots_adjust(hspace=0.4, wspace=0.3)
plt.show()


In [ ]:
# Outliers pelo método do IQR (geral)
q1 = df['frequencia_treino'].quantile(0.25)
q3 = df['frequencia_treino'].quantile(0.75)
iqr = q3 - q1

outliers = df[(df['frequencia_treino'] < (q1 - 1.5 * iqr)) | (df['frequencia_treino'] > (q3 + 1.5 * iqr))]
print(f"Total de outliers encontrados: {len(outliers)}")
outliers

In [ ]:
# Tabela descritiva do ganho de massa para a frequência de treino

freq_massa = df.groupby('frequencia_treino')['ganho_massa'].agg(['mean', 'median', 'std', 'count'])
freq_massa.columns = ['Média de Ganho (kg)', 'Mediana de Ganho (kg)', 'Desvio Padrão (kg)', 'Tamanho da Amostra']
freq_massa = freq_massa.round(4)
freq_massa

In [ ]:
# Ganho de massa para cada frequência de treino
# Visual da distribuição por Frequência de Treino reorganizado e com eixos padronizados

# Encontra os limites globais do ganho de massa para a padronização das escalas
ganho_min = df['ganho_massa'].min() - 0.5
ganho_max = df['ganho_massa'].max() + 0.5

limites_atleta = 18

# Descobre quantas frequências de treino diferentes existem para criar o número certo de colunas
frequencias_ordenadas = sorted(df['frequencia_treino'].unique())
num_colunas = len(frequencias_ordenadas)

fig, axes = plt.subplots(2, num_colunas, figsize=(5 * num_colunas, 12))

for i, freq in enumerate(frequencias_ordenadas):
  df_freq = df[df['frequencia_treino'] == freq]
  cor_atual = sns.color_palette('flare', num_colunas)[i]

  # ---Histogramas ---
  sns.histplot(data=df_freq, x='ganho_massa', kde=True, color=cor_atual, stat='count', kde_kws={'cut': 3}, ax=axes[0, i])
  axes[0, i].set_title(f'Histograma - Treino: {freq}x', fontsize=11, weight='bold')
  axes[0, i].set_xlabel('Ganho de Massa (kg)')
  axes[0, i].set_ylabel('Número de atletas')
  
  # Padronizando os eixos X e Y para todos os histogramas
  axes[0, i].set_xlim(ganho_min, ganho_max)
  axes[0, i].set_ylim(0, limites_atleta)

  # Adiciona os rótulos de contagem no histograma
  if axes[0, i].containers:
      axes[0, i].bar_label(axes[0, i].containers[0], padding=3, fontsize=9, weight='bold')

  # ---Boxplots ---
  sns.boxplot(data=df_freq, y='ganho_massa', color=cor_atual, ax=axes[1, i])
  sns.stripplot(data=df_freq, y='ganho_massa', color='black', alpha=0.4, size=5, jitter=0.1, ax=axes[1, i])
  axes[1, i].set_title(f'Boxplot - Treino: {freq}x', fontsize=11, weight='bold')
  axes[1, i].set_ylabel('Ganho de Massa (kg)')
  axes[1, i].set_xlabel(f'{freq}x por semana')
  
  # Padronizando o eixo Y de todos os Boxplots
  axes[1, i].set_ylim(ganho_min, ganho_max)

# Ajusta os espaçamentos
plt.tight_layout(pad=3.0)
plt.subplots_adjust(hspace=0.4, wspace=0.3)
plt.show()


In [ ]:
# Tabela descritiva do ganho de massa por idade

idade_massa = df.groupby('idade')['ganho_massa'].agg(['mean', 'median', 'std', 'count'])
idade_massa.columns = ['Média de Ganho (kg)', 'Mediana de Ganho (kg)', 'Desvio Padrão (kg)', 'Tamanho da Amostra']

idade_massa = idade_massa.round(4)
idade_massa

In [ ]:
# Dividindo a quantidade de idade em intervalos

# 1. Definir as três condições lógicas
condicoes = [
    (df['idade'] <= 24),                              # De 18 a 24 anos
    (df['idade'] >= 25) & (df['idade'] <= 31),        # De 25 a 31 anos
    (df['idade'] >= 32)                               # De 32 a 39 anos
]

# 2. Definir os rótulos correspondentes para cada condição
rotulos = ['18 a 24 anos', '25 a 31 anos', '32 a 39 anos']

# 3. Cria a coluna nova preenchida com um valor padrão
df['faixa_etaria'] = 'Não identificado'

for condicao, rotulo in zip(condicoes, rotulos):
    df.loc[condicao, 'faixa_etaria'] = rotulo

# 5. Verifica quantas pessoas caíram em cada grupo
df['faixa_etaria'].value_counts()

In [ ]:
# Tabela descritiva do ganho de massa para o intervalo de idade criado

idade_ganho = df.groupby('faixa_etaria')['ganho_massa'].agg(['mean', 'median', 'std', 'count'])
idade_ganho.columns = ['Média de Ganho (kg)', 'Mediana de Ganho (kg)', 'Desvio Padrão (kg)', 'Tamanho da Amostra']

idade_ganho = idade_ganho.round(4)
idade_ganho

In [ ]:
# Tabela descritiva da frequência de treino por idade

idade_treino = df.groupby('idade')['frequencia_treino'].agg(['mean', 'median', 'std', 'count'])
idade_treino.columns = ['Média de Treino', 'Mediana de Treino', 'Desvio Padrão', 'Tamanho da Amostra']

idade_treino = idade_treino.round(4)
idade_treino

In [ ]:
# Tabela descritiva para o intervalo de idade criado

idade_freq = df.groupby('faixa_etaria')['frequencia_treino'].agg(['mean', 'median', 'std', 'count'  ])
idade_freq.columns = ['Média de Frequência', 'Mediana de Frequência', 'Desvio Padrão', 'Tamanho da Amostra']

idade_freq = idade_freq.round(4)
idade_freq

### Questão 2 — Existe diferença estatisticamente significativa entre as fórmulas?

### Questão 3 — Correlação entre idade e ganho de massa muscular

### Questão 4 — Frequência de treino x ganho de massa (independente do suplemento)

### Questão 5 — Interação entre idade, frequência de treino e eficácia do suplemento

###  Questão 6 — Qual fórmula recomendar para atletas que treinam mais de 5x/semana?

### Visualizações - Gráficos

### Conclusão geral